In [1]:
!pip install -q youtube-transcript-api langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers bitsandbytes accelerate fastapi uvicorn

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"
USE_4BIT = True  

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
model.eval()


def generate_text(messages, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

In [3]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi


def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        vid = parsed.path.lstrip("/").split("/")[0]
        if vid:
            return vid

    if host in ("youtube.com", "m.youtube.com", "music.youtube.com"):
        qs = parse_qs(parsed.query)
        if qs.get("v"):
            return qs["v"][0]
        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "live", "v"):
            return parts[1]

    raise ValueError(f"No video id found in URL: {url}")


def load_segments(url: str, languages=("ar", "en")):
    """Returns list of (text, start_seconds)."""
    video_id = extract_video_id(url)
    fetched = YouTubeTranscriptApi().fetch(video_id, languages=list(languages))
    return [(s.text.replace("\n", " ").strip(), s.start) for s in fetched]

In [4]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "BAAI/bge-m3"  # عربي + إنجليزي


def build_documents(segments, max_chars=900, overlap_snips=2):
    docs, buf, buf_len, new_count = [], [], 0, 0
    for text, start in segments:
        if not text:
            continue
        buf.append((text, start))
        buf_len += len(text) + 1
        new_count += 1
        if buf_len >= max_chars:
            docs.append(Document(
                page_content=" ".join(t for t, _ in buf),
                metadata={"start": buf[0][1]},
            ))
            buf = buf[-overlap_snips:]
            buf_len = sum(len(t) + 1 for t, _ in buf)
            new_count = 0
    if new_count > 0 and buf:
        docs.append(Document(
            page_content=" ".join(t for t, _ in buf),
            metadata={"start": buf[0][1]},
        ))
    return docs


embedding = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

/tmp/ipykernel_355/5103343.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [5]:
def fmt_time(sec: float) -> str:
    sec = int(sec)
    return f"{sec // 3600:02d}:{(sec % 3600) // 60:02d}:{sec % 60:02d}"


RULES = (
    "You answer questions about a YouTube video using ONLY the transcript "
    "excerpts below. If the answer is not in the excerpts, say clearly that "
    "the video does not cover it. Never use outside knowledge. "
    "Answer in the same language as the question. Be concise and accurate."
)


def answer_question(db, question: str, k: int = 5):
    hits = db.similarity_search(question, k=k)
    hits = sorted(hits, key=lambda d: d.metadata["start"])
    context = "\n\n".join(
        f"[{fmt_time(d.metadata['start'])}] {d.page_content}" for d in hits
    )
    reply = generate_text([{
        "role": "user",
        "content": f"{RULES}\n\nTranscript excerpts:\n{context}\n\nQuestion: {question}",
    }])
    return reply, hits

In [10]:
import secrets
import threading
import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel

API_TOKEN = "change-me-to-a-long-random-string"  #API_TOKEN
PORT = 8001

app = FastAPI(title="YouTube RAG API")
INDEXES = {}                 
GPU_LOCK = threading.Lock()  


def check_token(x_api_key: str = Header(default="")):
    if not secrets.compare_digest(x_api_key.encode(), API_TOKEN.encode()):
        raise HTTPException(status_code=401, detail="Invalid API key")


class IndexReq(BaseModel):
    url: str
    manual_transcript: str = ""


class AskReq(BaseModel):
    video_id: str
    question: str
    k: int = 5


@app.get("/health")
def health():
    return {"status": "ok", "videos": list(INDEXES)}


@app.post("/index", dependencies=[Depends(check_token)])
def index_video(req: IndexReq):
    try:
        video_id = extract_video_id(req.url)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

    manual = req.manual_transcript.strip()
    if video_id in INDEXES and not manual:
        return {"video_id": video_id, "cached": True}

    if manual:
        segments = [(l.strip(), 0.0) for l in manual.splitlines() if l.strip()]
    else:
        try:
            segments = load_segments(req.url)
        except Exception as e:
            raise HTTPException(
                status_code=502,
                detail=f"Could not fetch transcript ({type(e).__name__}). "
                       "Paste it manually instead.",
            )

    docs = build_documents(segments)
    if not docs:
        raise HTTPException(status_code=422, detail="Transcript is empty")

    with GPU_LOCK:
        INDEXES[video_id] = FAISS.from_documents(docs, embedding)
    return {
        "video_id": video_id,
        "segments": len(segments),
        "chunks": len(docs),
        "manual": bool(manual),
        "cached": False,
    }


@app.post("/ask", dependencies=[Depends(check_token)])
def ask_video(req: AskReq):
    db = INDEXES.get(req.video_id)
    if db is None:
        raise HTTPException(status_code=404, detail="Video not indexed yet")

    with GPU_LOCK:
        reply, hits = answer_question(db, req.question, req.k)

    return {
        "answer": reply,
        "sources": [
            {
                "time": fmt_time(d.metadata["start"]),
                "seconds": int(d.metadata["start"]),
                "text": d.page_content[:220],
            }
            for d in hits
        ],
    }


threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
    daemon=True,
).start()
print("API running on port", PORT)

API running on port 8001


In [11]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = " "  #TOKEN NGROK

ngrok.kill()  
ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(PORT, bind_tls=True)
print("PUBLIC URL:", tunnel.public_url)

PUBLIC URL: https://photo-willpower-crayfish.ngrok-free.dev
